# Adult differential nucleosome positioning

Lineage-aware interrogation of adult scATAC nucleosome-dyad calls. The primary comparison is each target cell type against the leave-one-out consensus of related adult cells; the global adult consensus is retained as context.


## Configuration

Results load from `runs/adult_diffnuc` by default or from `CHROMBPNET_DIFFNUC_RESULT`. Choose a target cell type and ranked cached locus below.


In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import polars as pl


def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError(
        "Could not locate the chrombpnet project root from the current Jupyter directory."
    )


PROJECT_ROOT = find_project_root()
RESULT_DIR = Path(
    os.environ.get(
        "CHROMBPNET_DIFFNUC_RESULT",
        PROJECT_ROOT / "runs" / "adult_diffnuc",
    )
).expanduser()
TARGET_CELL = None
LOCUS_RANK = 0

calls_path = RESULT_DIR / "differential_nucleosomes.parquet"
summary_path = RESULT_DIR / "summary.json"
profiles_path = RESULT_DIR / "profiles.npz"
missing = [path for path in (calls_path, summary_path, profiles_path) if not path.is_file()]
if missing:
    missing_names = ", ".join(path.name for path in missing)
    raise FileNotFoundError(
        f"Adult differential-nucleosome results are not ready in {RESULT_DIR.resolve()}.\n"
        f"Missing: {missing_names}.\n"
        "Generate them with `uv run chrombpnet-call-diff-nucleosomes ...`, "
        "or set CHROMBPNET_DIFFNUC_RESULT to a completed result directory. "
        "See docs/adult_differential_nucleosomes.md for the full command."
    )

calls = pl.read_parquet(calls_path)
summary = json.loads(summary_path.read_text())
cache = np.load(profiles_path, allow_pickle=False)
cell_types = cache["cell_types"].astype(str)
lineages = cache["lineages"].astype(str)
locus_ids = cache["locus_ids"].astype(str)
centers = cache["centers"].astype(int)
chroms = cache["chroms"].astype(str)
raw_profiles = cache["raw"]
normalized_profiles = cache["normalized"]

cached_loci = set(locus_ids.tolist())
cached_calls = calls.filter(pl.col("locus_id").is_in(sorted(cached_loci)))
if TARGET_CELL is None:
    counts = (
        cached_calls.filter(pl.col("is_differential"))
        .group_by("cell_type")
        .len()
        .sort("len", descending=True)
    )
    TARGET_CELL = counts.row(0, named=True)["cell_type"] if counts.height else cell_types[0]

print(f"Result directory: {RESULT_DIR.resolve()}")
print(f"Adult cell types: {summary['cohort']['cell_types']}")
print(f"Tested loci: {summary['loci_tested']:,}")
print(f"Lineage-aware calls: {summary['lineage_calls']:,}")


## Adult cohort QC


In [ ]:
lineage_qc = pl.DataFrame({"cell_type": cell_types, "lineage": lineages}).group_by("lineage").len().sort("len", descending=True)
call_qc = (
    calls.group_by("lineage")
    .agg(
        pl.len().alias("tests"),
        pl.col("is_differential").sum().alias("lineage_calls"),
        pl.col("is_global_differential").sum().alias("global_calls"),
        pl.col("depth").median().alias("median_depth"),
    )
    .sort("lineage_calls", descending=True)
)
display(lineage_qc)
display(call_qc)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(lineage_qc["lineage"], lineage_qc["len"], color="#2b6f77")
axes[0].set_ylabel("Adult cell types")
axes[0].tick_params(axis="x", rotation=90)
axes[0].set_title("Cohort composition")
axes[1].bar(call_qc["lineage"], call_qc["lineage_calls"], color="#a64b2a")
axes[1].set_ylabel("Differential loci")
axes[1].tick_params(axis="x", rotation=90)
axes[1].set_title("Lineage-aware calls")
fig.tight_layout()


## Ranked differential sites


In [ ]:
def ranked_sites(target_cell):
    target = cached_calls.filter(pl.col("cell_type") == target_cell)
    ranked = target.filter(pl.col("is_differential")).sort(
        ["lineage_robust_z", "lineage_effect"], descending=True
    )
    if ranked.is_empty():
        ranked = target.sort(["lineage_robust_z", "lineage_effect"], descending=True)
    return ranked.with_row_index("rank")

ranked = ranked_sites(TARGET_CELL)
if ranked.is_empty():
    raise ValueError(f"No cached loci are available for {TARGET_CELL}")
LOCUS_RANK = min(max(int(LOCUS_RANK), 0), ranked.height - 1)
selected = ranked.row(LOCUS_RANK, named=True)
selected_locus = selected["locus_id"]
locus_index = int(np.flatnonzero(locus_ids == selected_locus)[0])
cell_index = int(np.flatnonzero(cell_types == TARGET_CELL)[0])
lineage = lineages[cell_index]
locus_depth = raw_profiles[locus_index].sum(axis=1)
qualified = locus_depth >= summary["parameters"]["min_counts"]
lineage_indices = np.flatnonzero(
    (lineages == lineage) & (cell_types != TARGET_CELL) & qualified
)
global_indices = np.flatnonzero((cell_types != TARGET_CELL) & qualified)
if len(lineage_indices) < summary["parameters"]["min_lineage_comparators"]:
    raise ValueError(f"{TARGET_CELL} has too few depth-qualified lineage comparators")

columns = [
    "rank", "locus_id", "cell_type", "lineage", "depth", "lineage_effect",
    "lineage_robust_z", "lineage_shift_bp", "split_half_noise",
    "global_effect", "global_robust_z", "global_shift_bp", "is_differential",
]
display(ranked.select(columns).head(30))
print(f"Selected locus: {selected_locus} | cell type: {TARGET_CELL} | lineage: {lineage}")


## Profile overlay


In [ ]:
x = np.arange(normalized_profiles.shape[-1]) - normalized_profiles.shape[-1] // 2
target_profile = normalized_profiles[locus_index, cell_index]
lineage_consensus = normalized_profiles[locus_index, lineage_indices].mean(axis=0)
global_consensus = normalized_profiles[locus_index, global_indices].mean(axis=0)
target_raw = raw_profiles[locus_index, cell_index]

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [1, 2]})
axes[0].vlines(x[target_raw > 0], 0, target_raw[target_raw > 0], color="#3b3b3b", linewidth=1)
axes[0].set_ylabel("Raw dyad count")
axes[0].set_title(f"{TARGET_CELL} at {selected_locus}")
axes[1].plot(x, target_profile, color="#b33a2b", linewidth=2.2, label=TARGET_CELL)
axes[1].plot(x, lineage_consensus, color="#26747a", linewidth=1.8, label=f"{lineage} consensus")
axes[1].plot(x, global_consensus, color="#555555", linewidth=1.4, linestyle="--", label="Adult global consensus")
axes[1].axvline(0, color="#9a9a9a", linewidth=0.8)
axes[1].set_xlabel("Position relative to candidate center (bp)")
axes[1].set_ylabel("Smoothed profile probability")
axes[1].legend(frameon=False)
fig.tight_layout()


## Target-minus-consensus deviation


In [ ]:
lineage_deviation = target_profile - lineage_consensus
global_deviation = target_profile - global_consensus

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(x, lineage_deviation, color="#b33a2b", linewidth=2, label="Target - lineage consensus")
ax.plot(x, global_deviation, color="#26747a", linewidth=1.5, label="Target - global consensus")
ax.axhline(0, color="#666666", linewidth=0.8)
ax.axvline(0, color="#9a9a9a", linewidth=0.8)
ax.fill_between(x, 0, lineage_deviation, where=lineage_deviation >= 0, color="#d97a5d", alpha=0.28)
ax.fill_between(x, 0, lineage_deviation, where=lineage_deviation < 0, color="#5d9da3", alpha=0.28)
ax.set_xlabel("Position relative to candidate center (bp)")
ax.set_ylabel("Probability deviation")
ax.set_title(
    f"Lineage effect={selected['lineage_effect']:.3f}, "
    f"z={selected['lineage_robust_z']:.2f}, shift={selected['lineage_shift_bp']} bp"
)
ax.legend(frameon=False)
fig.tight_layout()


## Cross-cell-type heatmap


In [ ]:
matrix = normalized_profiles[locus_index]
row_max = matrix.max(axis=1, keepdims=True)
scaled = np.divide(matrix, row_max, out=np.zeros_like(matrix), where=row_max > 0)
order = np.lexsort((cell_types, lineages, cell_types != TARGET_CELL))
ordered = scaled[order]

fig, ax = plt.subplots(figsize=(13, 10))
image = ax.imshow(
    ordered,
    aspect="auto",
    interpolation="nearest",
    cmap="magma",
    extent=[x[0], x[-1], len(order), 0],
    vmin=0,
    vmax=1,
)
step = max(1, len(order) // 24)
ticks = np.arange(0, len(order), step)
ax.set_yticks(ticks + 0.5)
ax.set_yticklabels([f"{cell_types[order[i]]} | {lineages[order[i]]}" for i in ticks], fontsize=8)
ax.axvline(0, color="white", linewidth=0.7, alpha=0.8)
ax.set_xlabel("Position relative to candidate center (bp)")
ax.set_ylabel("Adult cell type")
ax.set_title(f"Cross-cell-type dyad shape at {selected_locus}")
fig.colorbar(image, ax=ax, label="Within-cell profile / maximum")
fig.tight_layout()


## Shift and effect distributions


In [ ]:
called = calls.filter(pl.col("is_differential"))
finite_shifts = called["lineage_shift_bp"].drop_nulls().to_numpy()
effects = called["lineage_effect"].drop_nulls().to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(finite_shifts, bins=41, color="#26747a", edgecolor="white")
axes[0].axvline(selected["lineage_shift_bp"], color="#b33a2b", linewidth=2, label="Selected")
axes[0].set_xlabel("Lineage-relative positioning shift (bp)")
axes[0].set_ylabel("Called target-locus pairs")
axes[0].legend(frameon=False)
axes[1].hist(effects, bins=40, color="#a64b2a", edgecolor="white")
axes[1].axvline(selected["lineage_effect"], color="#26747a", linewidth=2, label="Selected")
axes[1].set_xlabel("Total-variation effect")
axes[1].set_ylabel("Called target-locus pairs")
axes[1].legend(frameon=False)
fig.tight_layout()

print(f"Genome-browser locus: {selected['chrom']}:{selected['start']}-{selected['end']}")
print(f"BED: {RESULT_DIR / 'beds' / (TARGET_CELL.replace('/', '_') + '.diffnuc.bed')}")


## Optional selectors

When `ipywidgets` is installed, these controls update `TARGET_CELL` and `LOCUS_RANK`. Rerun the ranked and visualization cells after changing a selector.


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display as display_widget

    cell_selector = widgets.Dropdown(
        options=sorted(cell_types.tolist()), value=TARGET_CELL, description="Cell type"
    )
    rank_selector = widgets.BoundedIntText(
        value=LOCUS_RANK, min=0, max=max(0, ranked.height - 1), description="Locus rank"
    )

    def _store_selection(change=None):
        global TARGET_CELL, LOCUS_RANK
        TARGET_CELL = cell_selector.value
        LOCUS_RANK = rank_selector.value

    cell_selector.observe(_store_selection, names="value")
    rank_selector.observe(_store_selection, names="value")
    display_widget(widgets.HBox([cell_selector, rank_selector]))
except ImportError:
    print("ipywidgets is not installed; edit TARGET_CELL and LOCUS_RANK in Configuration.")
